# New Section

In [ ]:
!pip install transformers datasets sentence-transformers torch pandas

In [ ]:
import pandas as pd
from transformers import pipeline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/menu.csv')
print(df.shape)
df.head()


In [ ]:
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli"
)

# Candidate labels from your dataset
candidate_labels = df["category"].unique().tolist()

# Example dish
dish = "Paneer Butter Masala"

result = classifier(
    dish,
    candidate_labels=candidate_labels
)

print(result)

In [ ]:
correct = 0
wrong = []

for _, row in df.iterrows():
    prediction = classifier(
        row["dish_name"],
        candidate_labels=candidate_labels,
        hypothesis_template="This menu item falls under the {} category."
    )
    predicted = prediction["labels"][0].strip().lower()
    actual = row["category"].strip().lower()

    if predicted == actual:
        correct += 1
    else:
        wrong.append({
            "dish": row["dish_name"],
            "actual": row["category"],
            "predicted": prediction["labels"][0]
        })

accuracy = correct / len(df)
print(f"Total dishes  : {len(df)}")
print(f"Correct       : {correct}")
print(f"Wrong         : {len(wrong)}")
print(f"Accuracy      : {accuracy:.2%}")
print("\n--- Mismatches ---")
for w in wrong:
    print(f"  {w['dish']:<30} | Actual: {w['actual']:<12} | Predicted: {w['predicted']}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def generate_description(dish_name, category, dietary_tags=""):
    tags_hint = f" It is {dietary_tags}." if dietary_tags else ""
    prompt = (
        f"You are a professional restaurant menu copywriter. "
        f"Write exactly one appetizing sentence (15-25 words) "
        f"describing '{dish_name}', a {category.lower()} item.{tags_hint} "
        f"Mention the cooking style and key flavors. "
        f"Do not start with the dish name. Do not repeat the prompt."
    )
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        **inputs,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.75,
        top_p=0.92,
        repetition_penalty=1.4
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

    if len(text.split()) < 8:
        text = (
            f"A carefully prepared {category.lower()} crafted with "
            f"premium ingredients and authentic flavors."
        )
    return text

In [ ]:
test_cases = [
    ("Chicken Biryani",    "Biryani",     "Non-Veg, Spicy"),
    ("Paneer Tikka",       "Starter",     "Veg, Spicy, Contains Dairy"),
    ("Gulab Jamun",        "Dessert",     "Veg, Contains Dairy"),
    ("Garlic Naan",        "Bread",       "Veg, Contains Dairy"),
    ("Mango Lassi",        "Beverage",    "Veg, Contains Dairy"),
]

print(f"{'Dish':<25} {'Category':<12} Generated Description")
print("-" * 85)
for dish, cat, tags in test_cases:
    desc = generate_description(dish, cat, tags)
    print(f"{dish:<25} {cat:<12} {desc}")

In [ ]:
def predict_category(dish_name):
    result = classifier(
        dish_name,
        candidate_labels=candidate_labels,
        hypothesis_template="This menu item falls under the {} category."
    )
    return result["labels"][0], round(result["scores"][0], 2)

def menu_assistant(dish_name):
    category, confidence = predict_category(dish_name)

    match = df[df["dish_name"].str.lower() == dish_name.lower()]
    tags = match["dietary_tags"].values[0] if len(match) > 0 else ""

    description = generate_description(dish_name, category, tags)

    return {
        "dish_name"            : dish_name,
        "predicted_category"   : category,
        "confidence"           : confidence,
        "generated_description": description
    }

In [ ]:
print("=" * 55)
print("       MenuManager — Smart Menu Assistant")
print("=" * 55)

while True:
    dish = input("\n Enter dish name (or 'quit' to exit): ").strip()
    if dish.lower() in ("quit", "exit", ""):
        print("\n Exiting MenuManager. Goodbye!")
        break
    result = menu_assistant(dish)
    print(f"\n  Dish       : {result['dish_name']}")
    print(f"  Category   : {result['predicted_category']}  (confidence: {result['confidence']})")
    print(f"  Description: {result['generated_description']}")
    print("-" * 55)

In [ ]:
def menu_assistant_safe(dish_name, threshold=0.45):
    category, confidence = predict_category(dish_name)

    if confidence < threshold:
        return {
            "dish_name": dish_name,
            "predicted_category": "Uncertain — please choose manually",
            "confidence": round(confidence, 2),
            "generated_description": "N/A"
        }

    match = df[df["dish_name"].str.lower() == dish_name.lower()]
    tags = match["dietary_tags"].values[0] if len(match) > 0 else ""
    description = generate_description(dish_name, category, tags)

    return {
        "dish_name": dish_name,
        "predicted_category": category,
        "confidence": round(confidence, 2),
        "generated_description": description
    }

# Test with ambiguous dish
print(menu_assistant_safe("veg fried rice"))
print(menu_assistant_safe("Chicken Biryani"))

gpt2

In [ ]:
from transformers import pipeline

gpt2 = pipeline(
    "text-generation",
    model="gpt2"
)

In [ ]:
prompt = "Restaurant description for Chicken Biryani."

result = gpt2(
    prompt,
    max_new_tokens=25
)

print(result[0]["generated_text"])

In [ ]:
while True:

    dish = input("Enter Dish Name (or exit): ")

    if dish.lower() == "exit":
        break

    category, confidence = predict_category(dish)

    description = generate_description(
        dish,
        category,
        ""
    )

    print("\nCategory :", category)
    print("Confidence :", round(confidence,2))
    print("Description :", description)

In [ ]:
def menu_assistant(dish_name, dietary_tags=""):

    category, confidence = predict_category(dish_name)

    description = generate_description(
        dish_name,
        category,
        dietary_tags
    )

    return {
        "Dish": dish_name,
        "Predicted Category": category,
        "Confidence": confidence,
        "Generated Description": description
    }

print(menu_assistant("Chicken Biryani"))